# Meta Quest VR + Head Tracking Collection Dashboard

이 노트북은 `/home/hyunjin/rby1_ws/rby1-data-collection/main.py` 에서 실행 중인 Meta Quest VR data collection backend에 붙는 얇은 대시보드입니다. 로봇 제어 로직을 여기서 직접 실행하는 것이 아니라, backend의 상태를 조회하고 녹화 명령을 보내는 용도입니다.

## 이 노트북에서 할 수 있는 것
- 현재 recording status 확인
- preflight readiness 확인
- torso-locked / bimanual ready pose 요청과 HMD head-tracking 상대영점 설정
- HMD yaw/pitch로 움직이는 `head_0/head_1` 상태 확인 (HMD position/roll은 무시)
- 카메라 진단 상태와 preview availability 확인
- 현재 저장 경로, 마지막 저장 경로, 마지막 discard 경로 확인
- Quest head-camera WebRTC streaming 상태 확인
- `Start`, `Stop & Save`, `Discard Last`, `Refresh` 명령 전송
- 저장된 H5에서 head joint와 torso 고정 상태 검증

## 권장 실행 순서
1. 별도 터미널에서 backend를 먼저 실행합니다.
   `python /home/hyunjin/rby1_ws/rby1-data-collection/main.py --data_collect --quest_video --dashboard_cmd_server tcp://127.0.0.1:5556`
2. 이 노트북에서 아래 셀들을 위에서 아래로 순서대로 실행합니다.
3. 중간에 endpoint를 바꿨다면 설정 셀만 다시 실행합니다.
4. UI 셀이 뜨면 `Init + Lock Torso` 또는 `Init Bimanual`로 ready pose를 요청합니다. 이동이 끝난 뒤 들어오는 첫 HMD frame이 head 상대영점이 되고, 그때의 torso pose는 세션 동안 고정됩니다.
5. 정면을 보고 초기화한 뒤 `Head Tracking`이 `TRACKING`인지 확인합니다. `STALE`이면 Quest pose packet 수신부터 확인합니다.
6. `Check Cameras` 또는 `Refresh`로 camera panel과 preview, Quest video 상태를 확인합니다.
7. preflight가 준비되면 `Start`로 녹화를 시작하고, HMD를 좌우/상하로 움직이며 데모를 수행한 뒤 `Stop & Save`를 누릅니다.
8. 마지막 H5 검증 셀에서 `head_0/head_1` 변화와 torso 6DOF의 고정 여부를 확인합니다.
9. 직전 데모를 버리려면 `Discard Last`를 사용합니다.

## 상태 해석
- `IDLE`은 backend가 살아 있지만 녹화를 아직 시작하지 않았다는 뜻입니다.
- `Ready pose`가 `WAIT`이면 아직 초기 자세 요청이 안 갔거나 로봇 이동이 끝나지 않은 상태입니다.
- `Head Tracking`의 `RECENTER WAIT`는 ready pose 이후 첫 fresh HMD frame을 기다리는 상태입니다.
- `STALE`이면 head는 마지막 목표에서 정지하고, packet이 복구되면 현재 head 위치에 자동 재영점됩니다.
- preview가 비어 있으면 camera diagnostics 패널의 `Live frames` 와 `Missing live`를 먼저 확인하면 됩니다.
- Quest 앱 안에 head-camera 영상을 보이려면 앱에 WebRTC receive-only 구현이 별도로 필요합니다. 이 노트북은 backend 상태만 표시합니다.

## 셀 구성
아래에 각 코드 셀의 역할을 설명하는 markdown을 붙여 두었습니다. 처음 보는 사람도 어떤 셀을 왜 실행하는지 바로 알 수 있게 정리한 것입니다.

## 셀 1. 기본 import와 backend client 로드

이 셀은 노트북이 동작하는 데 필요한 Python 모듈을 import하고, `rby1-data-collection` 경로를 `sys.path`에 추가한 뒤 dashboard client helper를 불러옵니다.

이 셀을 먼저 실행해야 이후 셀에서 `VRDashboardClient`, 상태 렌더링 함수, 위젯 구성이 정상적으로 동작합니다.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any
import html
import math
import sys
import threading
import time

import ipywidgets as widgets
from IPython.display import display

REPO_ROOT = Path("/home/hyunjin/rby1_ws/rby1-data-collection")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vr_dashboard_client import (
    VRDashboardClient,
    render_camera_html,
    render_paths_html,
    render_preflight_html,
    render_status_badge,
)

print(f"Repo root: {REPO_ROOT}")

## 셀 2. 연결 endpoint와 refresh 동작 설정

이 셀은 backend REP 서버 주소, 요청 타임아웃, 자동 새로고침 주기, preview 포함 여부 같은 런타임 설정을 정의합니다.

backend를 다른 포트나 다른 호스트에서 띄웠다면 여기의 `COMMAND_ENDPOINT`를 먼저 바꾸고 실행하면 됩니다.

In [ ]:
COMMAND_ENDPOINT = "tcp://127.0.0.1:5556"
REQUEST_TIMEOUT_MS = 4000
AUTO_REFRESH_SECS = 1.0
INCLUDE_PREVIEW = True
PREVIEW_MAX_WIDTH = "480px"

client = VRDashboardClient(COMMAND_ENDPOINT, timeout_ms=REQUEST_TIMEOUT_MS)
print(f"Dashboard endpoint: {COMMAND_ENDPOINT}")

## 셀 3. 대시보드 세션 상태 보관

이 셀은 notebook 쪽에서만 쓰는 `DashboardState`를 정의합니다. 마지막으로 받아온 snapshot, auto refresh 스레드, 중지 이벤트를 한 곳에 묶어 둡니다.

UI를 그리기 전에 이 상태 객체가 있어야 이후 helper 함수들이 공통 상태를 안전하게 공유할 수 있습니다.

In [ ]:
@dataclass(slots=True)
class DashboardState:
    last_snapshot: dict[str, Any] = field(default_factory=dict)
    auto_refresh_enabled: bool = False
    auto_refresh_stop: threading.Event = field(default_factory=threading.Event)
    auto_refresh_thread: threading.Thread | None = None


dashboard_state = DashboardState()
dashboard_state

## 셀 4. 렌더링과 command helper 정의

이 셀은 preview HTML 생성, backend snapshot 조회, 상태 패널 업데이트, `start`/`stop`/`discard` 명령 전송, auto refresh 루프를 담당하는 핵심 helper 함수들을 정의합니다.

실제 버튼 동작은 다음 UI 셀에서 연결되지만, 그 기반 로직은 모두 여기에서 준비됩니다.

In [ ]:
def render_preview_html(snapshot: dict[str, Any]) -> str:
    encoded = snapshot.get("preview_jpeg_base64")
    if not encoded:
        camera_diagnostics = snapshot.get("camera_diagnostics", {})
        detail = camera_diagnostics.get("message") or snapshot.get("preflight", {}).get("camera_message") or "No live camera frame is available yet."
        return (
            "<div style='padding:14px;border:1px dashed #9fb3c8;border-radius:12px;color:#486581;background:#f8fbfc;'>"
            "<div style='font-weight:700;margin-bottom:4px;'>Preview unavailable</div>"
            f"<div style='font-size:13px;'>{detail}</div>"
            "</div>"
        )
    return (
        "<div style='padding:6px;border:1px solid #d9e2ec;border-radius:12px;background:#ffffff;'>"
        f"<img src='data:image/jpeg;base64,{encoded}' style='width:100%;max-width:{PREVIEW_MAX_WIDTH};border-radius:10px;'/>"
        "</div>"
    )


def render_head_tracking_html(head_tracking: dict[str, Any]) -> str:
    if not head_tracking:
        return "<div style='padding:10px 14px;border:1px solid #d9e2ec;border-radius:12px;'>Head tracking status unavailable.</div>"

    recenter_pending = bool(head_tracking.get("recenter_pending", False))
    stale = bool(head_tracking.get("stale", True))
    calibrated = bool(head_tracking.get("calibrated", False))
    if recenter_pending:
        label = "RECENTER WAIT"
        color = "#d38b18"
    elif stale:
        label = "STALE / HOLD"
        color = "#a63d40"
    elif calibrated:
        label = "TRACKING"
        color = "#1b7f4a"
    else:
        label = "WAIT"
        color = "#556b75"

    target = head_tracking.get("target")
    if isinstance(target, (list, tuple)) and len(target) == 2:
        target_text = ", ".join(f"head_{idx}: {math.degrees(float(value)):+.1f}°" for idx, value in enumerate(target))
    else:
        target_text = "-"
    packet_age = head_tracking.get("packet_age_sec")
    packet_age_text = f"{float(packet_age):.3f}s" if isinstance(packet_age, (int, float)) else "-"
    return (
        "<div style='padding:10px 14px;border:1px solid #d9e2ec;border-radius:12px;background:#ffffff;'>"
        "<div style='font-size:12px;color:#486581;margin-bottom:6px;'>HEAD TRACKING · TORSO LOCKED</div>"
        f"<div style='font-size:14px;color:{color};font-weight:700;'>{label}</div>"
        f"<div style='font-size:13px;'><b>Target:</b> {target_text}</div>"
        f"<div style='font-size:13px;'><b>Packet age:</b> {packet_age_text}</div>"
        "<div style='font-size:12px;color:#486581;margin-top:6px;'>HMD position/roll ignored · yaw/pitch → head_0/head_1 only</div>"
        "</div>"
    )


def render_quest_video_html(video: dict[str, Any]) -> str:
    if not video:
        return "<div style='padding:10px 14px;border:1px solid #d9e2ec;border-radius:12px;'>Quest video status unavailable.</div>"

    enabled = bool(video.get("enabled", False))
    running = bool(video.get("running", False))
    peer_state = str(video.get("peer_state") or "disconnected")
    camera_running = bool(video.get("camera_running", False))
    error = video.get("error")
    if not enabled:
        label = "DISABLED"
        color = "#556b75"
    elif running:
        label = "RUNNING"
        color = "#1b7f4a"
    else:
        label = "ERROR"
        color = "#a63d40"
    frame_age = video.get("last_frame_age_sec")
    frame_age_text = f"{float(frame_age):.3f}s" if isinstance(frame_age, (int, float)) else "-"
    format_text = f"{video.get('codec', '-')} {video.get('width', '-')}x{video.get('height', '-')}@{video.get('fps', '-')}fps"
    error_html = f"<div style='font-size:12px;color:#a63d40;'>{html.escape(str(error))}</div>" if error else ""
    return (
        "<div style='padding:10px 14px;border:1px solid #d9e2ec;border-radius:12px;background:#ffffff;'>"
        "<div style='font-size:12px;color:#486581;margin-bottom:6px;'>QUEST HEAD-CAM VIDEO</div>"
        f"<div style='font-size:14px;color:{color};font-weight:700;'>{label}</div>"
        f"<div style='font-size:13px;'><b>Camera:</b> {'LIVE' if camera_running else 'WAIT'}</div>"
        f"<div style='font-size:13px;'><b>Peer:</b> {html.escape(peer_state)}</div>"
        f"<div style='font-size:13px;'><b>Format:</b> {html.escape(format_text)}</div>"
        f"<div style='font-size:13px;'><b>Frame age:</b> {frame_age_text}</div>"
        f"{error_html}</div>"
    )


def set_log(message: str):
    log_html.value = (
        "<div style='padding:8px 12px;border-radius:10px;background:#eef6ff;color:#102a43;'>"
        f"{message}"
        "</div>"
    )


def fetch_snapshot(include_preview: bool = INCLUDE_PREVIEW) -> dict[str, Any]:
    snapshot = client.get_status(include_preview=include_preview)
    dashboard_state.last_snapshot = snapshot
    return snapshot


def update_dashboard(snapshot: dict[str, Any]):
    has_state = isinstance(snapshot.get("preflight"), dict) and "status" in snapshot

    if snapshot.get("ok", False) or has_state:
        status_message = snapshot.get("message", "")
        if not snapshot.get("ok", False) and snapshot.get("error"):
            status_message = f"{status_message} | {snapshot['error']}" if status_message else snapshot["error"]

        status_html.value = render_status_badge(snapshot.get("status", "unknown"), status_message)
        preflight_html.value = render_preflight_html(snapshot.get("preflight", {}))
        camera_html.value = render_camera_html(snapshot.get("camera_diagnostics", {}))
        head_tracking_html.value = render_head_tracking_html(snapshot.get("head_tracking", {}))
        quest_video_html.value = render_quest_video_html(snapshot.get("quest_video", {}))
        paths_html.value = render_paths_html(snapshot)
        preview_html.value = render_preview_html(snapshot)
        raw_state.value = repr({k: v for k, v in snapshot.items() if k != "preview_jpeg_base64"})
    else:
        error_text = snapshot.get("error", "Unknown backend error")
        status_html.value = render_status_badge("error", error_text)
        preflight_html.value = render_preflight_html({"message": error_text})
        camera_html.value = render_camera_html({"message": error_text})
        head_tracking_html.value = render_head_tracking_html({})
        quest_video_html.value = render_quest_video_html({})
        paths_html.value = "<div style='padding:10px 14px;color:#9b2226;'>No session data available.</div>"
        preview_html.value = render_preview_html({})
        raw_state.value = repr(snapshot)


def refresh_dashboard(_=None):
    try:
        snapshot = fetch_snapshot(include_preview=INCLUDE_PREVIEW)
        update_dashboard(snapshot)
        set_log(f"Refreshed at {time.strftime('%H:%M:%S')}")
    except Exception as exc:
        update_dashboard({"ok": False, "error": str(exc)})
        set_log(f"Refresh failed: {exc}")


def run_command(command_name: str):
    try:
        if command_name == "start":
            snapshot = client.start_recording()
        elif command_name == "stop":
            snapshot = client.stop_recording()
        elif command_name == "discard":
            snapshot = client.discard_last_demo()
        elif command_name == "init_torso":
            snapshot = client.init_torso_mode()
        elif command_name == "init_bimanual":
            snapshot = client.init_bimanual_mode()
        elif command_name == "check_cameras":
            snapshot = client.check_cameras()
        else:
            raise ValueError(f"Unsupported command: {command_name}")

        dashboard_state.last_snapshot = snapshot
        update_dashboard(snapshot)

        command_message = snapshot.get("command_message")
        if not command_message:
            command_message = f"Command '{command_name}' sent at {time.strftime('%H:%M:%S')}"
        elif command_name in {"init_torso", "init_bimanual"}:
            command_message = f"{command_message} Refresh or Auto Refresh while the robot moves."

        set_log(command_message)
    except Exception as exc:
        update_dashboard({"ok": False, "error": str(exc)})
        set_log(f"Command '{command_name}' failed: {exc}")


def stop_auto_refresh():
    dashboard_state.auto_refresh_enabled = False
    dashboard_state.auto_refresh_stop.set()


def start_auto_refresh():
    if dashboard_state.auto_refresh_thread is not None and dashboard_state.auto_refresh_thread.is_alive():
        return

    dashboard_state.auto_refresh_enabled = True
    dashboard_state.auto_refresh_stop.clear()

    def _loop():
        while not dashboard_state.auto_refresh_stop.is_set():
            refresh_dashboard()
            dashboard_state.auto_refresh_stop.wait(AUTO_REFRESH_SECS)

    dashboard_state.auto_refresh_thread = threading.Thread(target=_loop, daemon=True)
    dashboard_state.auto_refresh_thread.start()

## 셀 5. 위젯 UI 생성과 첫 상태 조회

이 셀은 ready pose 버튼, camera check 버튼, recording 버튼, 상태 패널, camera diagnostics, preview, raw state 영역을 실제 `ipywidgets` 레이아웃으로 만들고 화면에 표시합니다. 셀 마지막의 `refresh_dashboard()`가 첫 backend 상태 조회를 즉시 수행합니다.

추가로 UI 상단에 각 dashboard 버튼이 Meta Quest VR 컨트롤러의 어떤 버튼과 대응되는지 한눈에 볼 수 있는 매핑 박스도 함께 표시합니다.

보통 사용자는 여기까지 실행한 뒤 `Init + Lock Torso` 또는 `Init Bimanual`로 초기 자세를 맞추고, head tracking과 camera/video panel을 확인한 뒤 녹화를 시작하게 됩니다.

In [ ]:
title_html = widgets.HTML("<h2 style='margin:4px 0'>Meta Quest VR Data Collection</h2>")
subtitle_html = widgets.HTML(
    "<div style='color:#486581;font-size:13px;'>Torso-locked head tracking, camera/video diagnostics, and recording controls for main.py</div>"
)
vr_mapping_html = widgets.HTML(
    """
    <div style='padding:10px 14px;border:1px solid #d9e2ec;border-radius:12px;background:#fffaf0;'>
      <div style='font-size:12px;color:#7c5e10;margin-bottom:6px;'>VR CONTROLLER MAP</div>
      <table style='border-collapse:collapse;font-size:13px;width:100%;'>
        <tr><td style='padding:4px 8px;'><b>Init + Lock Torso</b></td><td style='padding:4px 8px;'>Right controller A</td></tr>
        <tr><td style='padding:4px 8px;'><b>Init Bimanual</b></td><td style='padding:4px 8px;'>Left controller Y</td></tr>
        <tr><td style='padding:4px 8px;'><b>Head yaw / pitch</b></td><td style='padding:4px 8px;'>Quest HMD orientation (automatic after init)</td></tr>
        <tr><td style='padding:4px 8px;'><b>Start</b></td><td style='padding:4px 8px;'>Left controller X toggle from IDLE</td></tr>
        <tr><td style='padding:4px 8px;'><b>Stop &amp; Save</b></td><td style='padding:4px 8px;'>Right controller B or Left controller X toggle while recording</td></tr>
        <tr><td style='padding:4px 8px;'><b>Check Cameras</b></td><td style='padding:4px 8px;'>Dashboard only</td></tr>
        <tr><td style='padding:4px 8px;'><b>Refresh / Auto Refresh</b></td><td style='padding:4px 8px;'>Dashboard only</td></tr>
        <tr><td style='padding:4px 8px;'><b>Discard Last</b></td><td style='padding:4px 8px;'>Dashboard only</td></tr>
      </table>
      <div style='font-size:12px;color:#486581;margin-top:8px;'>
        Hold the matching controller grip button to follow that arm. Use the matching trigger to open or close each gripper.
        HMD motion never moves the torso; the ready-pose torso transform remains locked.
      </div>
    </div>
    """
)
status_html = widgets.HTML()
preflight_html = widgets.HTML()
camera_html = widgets.HTML()
head_tracking_html = widgets.HTML()
quest_video_html = widgets.HTML()
paths_html = widgets.HTML()
preview_html = widgets.HTML()
log_html = widgets.HTML()
raw_state = widgets.Textarea(
    value="",
    description="Raw",
    layout=widgets.Layout(width="100%", height="180px"),
)

btn_init_torso = widgets.Button(description="Init + Lock Torso", button_style="primary", icon="send", layout=widgets.Layout(width="170px"))
btn_init_bimanual = widgets.Button(description="Init Bimanual", button_style="primary", icon="send")
btn_check_cameras = widgets.Button(description="Check Cameras", button_style="info", icon="camera")
btn_refresh = widgets.Button(description="Refresh", button_style="info", icon="refresh")
auto_refresh = widgets.ToggleButton(description="Auto Refresh", value=False, icon="repeat")
btn_start = widgets.Button(description="Start", button_style="success", icon="play")
btn_stop = widgets.Button(description="Stop & Save", button_style="warning", icon="stop")
btn_discard = widgets.Button(description="Discard Last", button_style="danger", icon="trash")

btn_init_torso.tooltip = "VR mapping: Right controller A; locks torso and recenters head"
btn_init_bimanual.tooltip = "VR mapping: Left controller Y"
btn_check_cameras.tooltip = "Dashboard only"
btn_refresh.tooltip = "Dashboard only"
auto_refresh.tooltip = "Dashboard only"
btn_start.tooltip = "VR mapping: Left controller X toggle from idle"
btn_stop.tooltip = "VR mapping: Right controller B or Left controller X toggle while recording"
btn_discard.tooltip = "Dashboard only"

btn_init_torso.on_click(lambda _: run_command("init_torso"))
btn_init_bimanual.on_click(lambda _: run_command("init_bimanual"))
btn_check_cameras.on_click(lambda _: run_command("check_cameras"))
btn_start.on_click(lambda _: run_command("start"))
btn_stop.on_click(lambda _: run_command("stop"))
btn_discard.on_click(lambda _: run_command("discard"))
btn_refresh.on_click(refresh_dashboard)


def _on_auto_refresh(change):
    if change["new"]:
        start_auto_refresh()
        set_log("Auto refresh enabled")
    else:
        stop_auto_refresh()
        set_log("Auto refresh disabled")


auto_refresh.observe(_on_auto_refresh, names="value")

init_controls = widgets.HBox([btn_init_torso, btn_init_bimanual, btn_check_cameras, btn_refresh, auto_refresh])
record_controls = widgets.HBox([btn_start, btn_stop, btn_discard])
controls = widgets.VBox([init_controls, record_controls])
left_panel = widgets.VBox([title_html, subtitle_html, vr_mapping_html, status_html, preflight_html, head_tracking_html, quest_video_html, camera_html, paths_html, log_html])
right_panel = widgets.VBox([widgets.HTML("<b>Camera Preview</b>"), preview_html, raw_state])
layout = widgets.VBox([controls, widgets.HBox([left_panel, right_panel])])

display(layout)
refresh_dashboard()

## 셀 6. wiring sanity check와 backend 연결 확인

이 셀은 helper 함수와 client가 정상적으로 연결되어 있는지 빠르게 점검합니다. backend가 이미 떠 있으면 현재 status, preflight, head tracking, Quest video, preview 가능 여부를 출력하고, 아직 안 떠 있으면 어떤 명령으로 띄워야 하는지 다시 안내합니다.

문제가 생겼을 때는 이 셀의 출력이 가장 빠른 1차 진단 포인트입니다.

In [ ]:
assert hasattr(client, "get_status")
assert callable(refresh_dashboard)
assert callable(run_command)

print("Dashboard helper wiring is ready.")
try:
    snapshot = client.get_status(include_preview=True)
    print("Backend reachable:", snapshot.get("ok", False))
    print("Status:", snapshot.get("status"))
    print("Preflight:", snapshot.get("preflight", {}).get("message"))
    print("Camera diagnostics:", snapshot.get("camera_diagnostics", {}).get("message"))
    print("Head tracking:", snapshot.get("head_tracking", {}))
    print("Quest video:", snapshot.get("quest_video", {}))
    print("Preview available:", bool(snapshot.get("preview_jpeg_base64")))
except Exception as exc:
    print("Backend is not reachable yet:", exc)
    print("Start the backend with:")
    print(f"  python /home/hyunjin/rby1_ws/rby1-data-collection/main.py --data_collect --quest_video --dashboard_cmd_server {COMMAND_ENDPOINT}")

## 셀 7. 저장된 H5에서 head joint 기록 검증

`Stop & Save` 후 이 셀을 실행하면 마지막으로 저장한 H5의 전체 joint vector를 검사합니다. 현재 RBY1 joint order에서 `head_0/head_1`은 index 0/1, torso 6DOF는 index 2~7입니다.

- head 두 joint가 설정한 최소 각도 이상 움직였는지 확인합니다.
- torso 여섯 joint의 peak-to-peak drift가 허용 오차 안인지 확인합니다.
- `robot_target_joints`도 같은 전체 joint width로 저장됐는지 확인합니다.
- head RGB frame 수와 shape를 함께 출력합니다.

다른 파일을 확인하려면 다음 코드 셀의 `H5_PATH`에 직접 경로를 지정하세요.

In [ ]:
import h5py
import numpy as np

# None이면 dashboard가 기억하는 마지막 저장 파일을 사용합니다.
H5_PATH: str | Path | None = None
HEAD_JOINT_INDICES = np.array([0, 1], dtype=int)
TORSO_JOINT_INDICES = np.arange(2, 8, dtype=int)
EXPECTED_RBY1_JOINT_COUNT = 22
HEAD_MOVEMENT_THRESHOLD_DEG = 1.0
TORSO_LOCK_TOLERANCE_DEG = 1.0


def resolve_h5_path(explicit_path: str | Path | None) -> Path:
    if explicit_path is not None:
        return Path(explicit_path).expanduser()

    snapshot = dashboard_state.last_snapshot
    saved_path = snapshot.get("last_saved_path")
    if not saved_path:
        snapshot = client.get_status(include_preview=False)
        dashboard_state.last_snapshot = snapshot
        saved_path = snapshot.get("last_saved_path")
    if not saved_path:
        raise RuntimeError("저장된 demo 경로가 없습니다. Stop & Save 후 Refresh하거나 H5_PATH를 직접 지정하세요.")
    return Path(saved_path).expanduser()


h5_path = resolve_h5_path(H5_PATH)
if not h5_path.is_file():
    raise FileNotFoundError(h5_path)

with h5py.File(h5_path, "r") as h5_file:
    if "samples/robot_position" not in h5_file:
        raise KeyError("H5에 /samples/robot_position이 없습니다.")

    robot_position = np.asarray(h5_file["samples/robot_position"], dtype=float)
    target_dataset = h5_file.get("samples/robot_target_joints")
    robot_target = None if target_dataset is None else np.asarray(target_dataset, dtype=float)
    head_rgb_dataset = h5_file.get("head_rgb/image")
    head_rgb_shape = None if head_rgb_dataset is None else tuple(head_rgb_dataset.shape)

if robot_position.ndim != 2 or robot_position.shape[0] < 2:
    raise ValueError(f"robot_position은 최소 두 sample의 2D 배열이어야 합니다: {robot_position.shape}")
if robot_position.shape[1] <= int(TORSO_JOINT_INDICES[-1]):
    raise ValueError(f"head/torso index를 검사하기에 joint width가 부족합니다: {robot_position.shape}")

head_position_deg = np.rad2deg(robot_position[:, HEAD_JOINT_INDICES])
torso_position_deg = np.rad2deg(robot_position[:, TORSO_JOINT_INDICES])
head_span_deg = np.ptp(head_position_deg, axis=0)
torso_span_deg = np.ptp(torso_position_deg, axis=0)
head_motion_recorded = bool(np.any(head_span_deg >= HEAD_MOVEMENT_THRESHOLD_DEG))
torso_locked = bool(np.max(torso_span_deg) <= TORSO_LOCK_TOLERANCE_DEG)
joint_width_ok = robot_position.shape[1] == EXPECTED_RBY1_JOINT_COUNT
target_width_ok = robot_target is not None and robot_target.ndim == 2 and robot_target.shape[1] == robot_position.shape[1]

print(f"H5: {h5_path}")
print(f"Samples / joint width: {robot_position.shape[0]} / {robot_position.shape[1]}")
print(f"[{'PASS' if joint_width_ok else 'CHECK'}] expected RBY1 joint width: {EXPECTED_RBY1_JOINT_COUNT}")
print(f"[{'PASS' if target_width_ok else 'CHECK'}] robot_target_joints contains the same full joint vector")
print(f"[{'PASS' if head_motion_recorded else 'CHECK'}] head span [deg]: {np.round(head_span_deg, 3).tolist()}")
print(f"[{'PASS' if torso_locked else 'CHECK'}] torso max drift [deg]: {float(np.max(torso_span_deg)):.3f}")
print(f"Head RGB dataset shape: {head_rgb_shape}")

if robot_target is not None and target_width_ok:
    target_head_span_deg = np.ptp(np.rad2deg(robot_target[:, HEAD_JOINT_INDICES]), axis=0)
    print(f"Target head span [deg]: {np.round(target_head_span_deg, 3).tolist()}")
if not head_motion_recorded:
    print("주의: head 움직임이 기준보다 작습니다. Quest가 TRACKING인지 확인하고 HMD를 좌우/상하로 움직여 다시 수집하세요.")
if not torso_locked:
    print("주의: torso drift가 허용 오차보다 큽니다. ready pose 재초기화와 torso command 상태를 확인하세요.")